# 🚀 RAMPRS AI Mirror — Cloud GPU Model Server

> **High-Quality AI Virtual Try-On and Clothing Photo Editor Backend**
> Powered by **Stable Diffusion v1-5 + ControlNet Canny + OpenCV YuNet Face-Lock** on **NVIDIA T4 GPU**.

### 📋 Instructions:
1. Click **Runtime** ➔ **Change runtime type** ➔ Select **T4 GPU** ➔ Click **Save**.
2. Run all cells below (`Ctrl + F9` or click the play button on each cell).
3. Copy the generated **Public HTTPS Cloudflare Tunnel URL** and paste it into **`⚙️ AI Server`** at [ramprsai.vercel.app](https://ramprsai.vercel.app)!

In [ ]:
# ── Step 1: Verify NVIDIA GPU Acceleration ────────────────────────
!nvidia-smi

In [ ]:
# ── Step 2: Clone RAMPRS Repository & Install Dependencies ─────────
import os
if not os.path.exists('RAMPRS'):
    !git clone https://github.com/rohitsagarpatra76-wq/RAMPRS.git

%cd /content/RAMPRS/ramprs-ai-mirror-mvp
!pip install -q diffusers transformers accelerate fastapi uvicorn pillow opencv-python

In [ ]:
# ── Step 3: Install Cloudflare Tunnel & Launch AI GPU Server ────────
import os, time, subprocess, re

# 1. Download cloudflared linux binary if not present
if not os.path.exists('/usr/local/bin/cloudflared'):
    print('[1/3] Downloading Cloudflare Tunnel binary...')
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
    !chmod +x /usr/local/bin/cloudflared

# 2. Launch FastAPI Model Server in background
print('[2/3] Launching Stable Diffusion + YuNet AI Server on GPU (Port 8000)...')
server_proc = subprocess.Popen(
    ['python3', '-m', 'uvicorn', 'model_server:app', '--host', '0.0.0.0', '--port', '8000'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
)

# Wait 4 seconds for server to bind
time.sleep(4)

# 3. Launch Cloudflare Tunnel to expose port 8000 with public HTTPS
print('[3/3] Requesting secure public HTTPS Cloudflare Tunnel...')
tunnel_proc = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8000'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
)

tunnel_url = None
for line in tunnel_proc.stdout:
    match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if match:
        tunnel_url = match.group(0)
        break

print('\n' + '='*72)
print('🎉 RAMPRS AI CLOUD GPU SERVER IS RUNNING ON NVIDIA GPU!')
print('='*72)
print(f'🔗 Public AI Endpoint:  {tunnel_url}')
print('='*72)
print('\n👉 Next Step (10 seconds):')
print('1. Open your Vercel website:  https://ramprsai.vercel.app')
print('2. Click "⚙️ AI Server" in the top navigation bar')
print(f'3. Paste your Public AI Endpoint:  {tunnel_url}')
print('4. Click "Save" — the status will turn GREEN: "AI Ready (CUDA)"!')
print('='*72 + '\n')

# Keep stream open to display real-time inference logs
for line in server_proc.stdout:
    print(line, end='')
